<a href="https://colab.research.google.com/github/Janya-Sharma-22/Project-02_natural-language-processing/blob/main/2301201220_proj02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install -q spacy scikit-learn nltk
!python -m spacy download en_core_web_sm

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 44.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [21]:
import re
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string

nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    """Simple preprocessing: lowercase, tokenize, remove stopwords & punctuation, lemmatize"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and t not in string.punctuation
    ]
    return " ".join(tokens)

In [31]:
# Sample dataset
data = [

    ("Where is my order #12345?", "order_status"),
    ("Track my order 98765", "order_status"),
    ("Order status for #99999", "order_status"),
    ("Has my package shipped?", "order_status"),
    ("Where is my order", "order_status"),

    ("How can I return a product?", "return_policy"),
    ("What's your return policy?", "return_policy"),
    ("I want to return an item", "return_policy"),
    ("How many days to return", "return_policy"),

    ("Does this phone support fast charging?", "product_info"),
    ("Is this available in blue?", "product_info"),
    ("Product warranty details", "product_info"),
    ("What is the battery capacity?", "product_info"),

    ("I want to cancel order #12345", "cancel_order"),
    ("Cancel my order 43210", "cancel_order"),

    ("When will I get my refund for order #7890?", "refund_status"),
    ("Refund for order 11111", "refund_status"),

    ("Hello", "greet"),
    ("Hi there", "greet"),
    ("Thanks", "thanks"),
    ("Thank you", "thanks"),
]

df = pd.DataFrame(data, columns=["text", "intent"])
df["processed"] = df["text"].apply(preprocess)
df

,text,intent,processed
0,Where is my order #12345?,order_status,order 12345
1,Track my order 98765,order_status,track order 98765
2,Order status for #99999,order_status,order status 99999
3,Has my package shipped?,order_status,package shipped
4,Where is my order,order_status,order
5,How can I return a product?,return_policy,return product
6,What's your return policy?,return_policy,'s return policy
7,I want to return an item,return_policy,want return item
8,How many days to return,return_policy,many day return
9,Does this phone support fast charging?,product_info,phone support fast charging


In [23]:
X = df["processed"]
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42, stratify=y)


intent_clf = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=42)
)

intent_clf.fit(X_train, y_train)

y_pred = intent_clf.predict(X_test)
print("Classification report:\n")
print(classification_report(y_test, y_pred))

Classification report:

               precision    recall  f1-score   support

 cancel_order       0.00      0.00      0.00         1
        greet       0.00      0.00      0.00         1
 order_status       0.60      1.00      0.75         3
 product_info       0.50      1.00      0.67         2
refund_status       0.00      0.00      0.00         1
return_policy       1.00      1.00      1.00         2
       thanks       0.00      0.00      0.00         1

     accuracy                           0.64        11
    macro avg       0.30      0.43      0.35        11
 weighted avg       0.44      0.64      0.51        11



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [24]:
ORDER_REGEX = re.compile(r"(?:#\s*\d+|\b\d{4,}\b)")

def extract_order_number(text):
    """Return a normalized order number string if found, else None"""
    if not text:
        return None
    m = ORDER_REGEX.search(text)
    if m:
        raw = m.group(0)
        norm = raw.replace(" ", "").lstrip("#")
        return norm
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ("CARDINAL","ORDINAL"):
            token = ent.text.strip().lstrip("#")
            if token.isdigit() and len(token) >= 4:
                return token
    return None

tests = [
    "Where is my order #12345?",
    "Track order 98765",
    "I ordered it - order 4321",
    "Any update?"
]
for t in tests:
    print(t, "->", extract_order_number(t))


Where is my order #12345? -> 12345
Track order 98765 -> 98765
I ordered it - order 4321 -> 4321
Any update? -> None


In [25]:
canned_texts = []
canned_map = []
for intent, templates in TEMPLATES.items():
    for i, temp in enumerate(templates):
        text = re.sub(r"\{.*?\}", "", temp).strip()
        if text:
            canned_texts.append(text)
            canned_map.append((intent, i))

fallback_vectorizer = TfidfVectorizer().fit(canned_texts)
canned_vectors = fallback_vectorizer.transform(canned_texts)

from sklearn.metrics.pairwise import cosine_similarity

def fallback_by_similarity(query, top_k=1):
    q_proc = preprocess(query)
    qv = fallback_vectorizer.transform([q_proc])
    sims = cosine_similarity(qv, canned_vectors).flatten()
    idx = sims.argmax()
    score = sims[idx]
    intent, template_idx = canned_map[idx]
    return intent, template_idx, float(score)


In [26]:
import random

def generate_response(user_text, confidence_threshold=0.4):
    """
    Steps:
    1) Preprocess & predict intent with classifier
    2) Extract order number (if any)
    3) If intent confidence low, use similarity fallback
    4) Return templated response (insert order id where needed)
    """
    if not user_text or not user_text.strip():
        return "Please type a question."

    processed = preprocess(user_text)
    try:
        probs = intent_clf.predict_proba([processed])[0]
        intents = intent_clf.named_steps['logisticregression'].classes_
        best_idx = probs.argmax()
        predicted_intent = intents[best_idx]
        confidence = probs[best_idx]
    except Exception:
        predicted_intent = intent_clf.predict([processed])[0]
        confidence = 1.0

    order_id = extract_order_number(user_text)

    if confidence < confidence_threshold:
        fb_intent, fb_temp_idx, sim_score = fallback_by_similarity(user_text)
        if sim_score > 0.3:
            predicted_intent = fb_intent
            confidence = sim_score


    templates = TEMPLATES.get(predicted_intent, TEMPLATES["fallback"])
    template = random.choice(templates)

    if "{order_id}" in template:
        if order_id:
            response = template.format(order_id=order_id)
        else:
            response = "Could you please provide your order number (e.g. #12345)?"
    else:
        response = template

    if predicted_intent == "fallback" and confidence < 0.2:
        response = "Sorry — I didn't get that. Can you rephrase or provide more details?"

    return response


In [30]:
print("Customer Support Chatbot (type 'exit' to quit)\n")
while True:
    user = input("You: ")
    if not user:
        continue
    if user.lower().strip() in ("exit", "quit"):
        print("Bot: Goodbye!")
        break
    bot = generate_response(user)
    print("Bot:", bot)


Customer Support Chatbot (type 'exit' to quit)

You: Where is my order #12345?
Bot: Order 12345 has been shipped and will arrive soon.
You: return?
Bot: Returns are accepted within 15 days with original packaging and invoice.
You: What's your return policy?
Bot: Returns are accepted within 15 days with original packaging and invoice.
You: Is this available in blue?
Bot: Yes — this product supports fast charging.
You: exit
Bot: Goodbye!
